# Tanager Mangrove Mapping - 01 Preprocessing

| | |
|---|---|
| **Authors**     | Muhammad Wahyu Ramadhan, Athar Abdurrahman B., Diniyarti |
| **Competition** | Planet Tanager Open Data Competition 2026 |
| **Topic**       | Transferable Mangrove Extent and Biomass Mapping Using Adaptive Spectral Thresholds |
| **Date**        | June 2026 |

---

**Scope:** HDF5 structure inspection, band extraction, spectral index computation, adaptive threshold calibration per scene.

## 0. Environment Setup

In [ ]:
# Install dependencies (commented out for production)
!pip install h5py xarray rioxarray rasterio geopandas scipy matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path
from scipy.ndimage import distance_transform_edt

import numpy as np
import matplotlib.pyplot as plt
import h5py

# ============================================================
# Project root — adjust per environment
# ============================================================
# Google Colab (Google Drive mounted)
ROOT = Path('/content/drive/MyDrive/PROJECT/Planet Tanager Competition 2026/tanager-mangrove-mapping')

# Local (uncomment if running locally)
# ROOT = Path('..').resolve()

DATA_RAW  = ROOT / 'data' / 'raw'
DATA_PROC = ROOT / 'data' / 'processed'
DATA_AOI  = ROOT / 'data' / 'aoi'
SRC       = ROOT / 'src'

sys.path.insert(0, str(ROOT))
from src.preprocessing import (
    inspect_hdf5,
    load_hdf5,
    hdf5_to_geotiff,
    load_geotiff_bands,
    compute_all_indices,
    compute_water_mask,
    compute_coastal_candidate_mask,
    apply_adaptive_threshold,
    RELEVANT_WAVELENGTHS
)

print(f'ROOT      : {ROOT}')
print(f'DATA_RAW  : {DATA_RAW}')
print(f'DATA_PROC : {DATA_PROC}')

In [ ]:
import importlib, src.preprocessing as _pre
importlib.reload(_pre)
import src.continuum as _cont
importlib.reload(_cont)
from src.preprocessing import (
    inspect_hdf5, load_hdf5, hdf5_to_geotiff, load_geotiff_bands,
    compute_all_indices, compute_water_mask, compute_coastal_candidate_mask,
    apply_adaptive_threshold, RELEVANT_WAVELENGTHS
)
from src.continuum import compute_reip
print("Reloaded.")

## 1. Scene Inventory

In [ ]:
# ============================================================
# Scene IDs — primary site first
# ============================================================
SCENES = {
    'sangatta'   : '20250302_030003_92_4001',
    'gujarat'    : '20250311_061550_53_4001',
    'elsalvador' : '20250223_165546_32_4001',
    'belize'     : '20250824_171857_84_4001',
    'australia'  : '20250608_014315_58_4001',
}

for site, sid in SCENES.items():
    h5_path = DATA_RAW / f'{site}_{sid}_ortho_sr_hdf5.h5'
    status  = 'OK' if h5_path.exists() else 'MISSING'
    print(f'  {site:<12}: {status}  ({h5_path.name})')

## 2. HDF5 Structure Inspection — Sangatta

In [ ]:
# ============================================================
# Inspect HDF5 keys, shape, wavelength array
# Run once — confirm structure before running hdf5_to_geotiff()
# ============================================================
def h5_path(site):
    return DATA_RAW / f'{site}_{SCENES[site]}_ortho_sr_hdf5.h5'

inspect_hdf5(str(h5_path('sangatta')))

# TODO: after inspection, update load_hdf5() in src/preprocessing.py
# to match actual HDF5 structure (REFLECTANCE_PATH, WAVELENGTH_PATH, etc.)

## 3. HDF5 → GeoTIFF Conversion

**Band selection rationale**

Tanager provides 426 hyperspectral bands across 380-2500 nm (~5 nm step). For computing the 5 mangrove-relevant indices, single bands are selected at the wavelengths defined in each index's original publication:

| Band | Wavelength (nm) | Used in | Reference |
|---|---|---|---|
| Green | 560 | MNDWI, MVI | Xu (2006); Baloloy et al. (2020) |
| Red | 660 | SAVI | Huete (1988) |
| NIR | 860 | NDMI, MVI, SAVI, EMI | Gao (1996); Baloloy et al. (2020); Huete (1988); Rahmila et al. (2026) |
| SWIR1 | 1640 | NDMI, MNDWI, MVI | Gao (1996); Xu (2006); Baloloy et al. (2020) |
| SWIR2 | 2200 | EMI | Rahmila et al. (2026) |

Rationale: wavelength targets are anchored to index definitions rather than to multispectral sensor band passes (Landsat, Sentinel-2), avoiding arbitrary cross-sensor harmonization while preserving direct linkage to peer-reviewed index formulations.

**References (DOI):**
- Baloloy et al. (2020) — 10.1016/j.isprsjprs.2020.06.001
- Gao (1996) — 10.1016/S0034-4257(96)00067-3
- Huete (1988) — 10.1016/0034-4257(88)90106-X
- Rahmila et al. (2026) — 10.1080/21580103.2026.2616443
- Xu (2006) — 10.1080/01431160600589179

In [ ]:
with h5py.File(str(h5_path('sangatta')), 'r') as f:
    raw = f['HDFEOS/GRIDS/HYP/Data Fields/surface_reflectance']
    print(f'  Dtype    : {raw.dtype}')
    print(f'  Raw min  : {raw[0].min():.4f}')
    print(f'  Raw max  : {raw[0].max():.4f}')
    print(f'  Raw mean : {raw[0].mean():.4f}')

In [ ]:
# ============================================================
# Convert all scenes — relevant bands only
# Output: data/processed/{site}_{scene_id}_{band}_{wl}nm.tif
# Run once — skip if already converted
# ============================================================
print(f'Bands to export: {RELEVANT_WAVELENGTHS}\n')

for site, scene_id in SCENES.items():
    hp = h5_path(site)
    if not hp.exists():
        print(f'  {site:<12}: SKIP (HDF5 not found)')
        continue

    print(f'  {site} — converting...')
    out = hdf5_to_geotiff(
        hdf5_path  = str(hp),
        output_dir = str(DATA_PROC),
        scene_id   = scene_id,
        site       = site
    )
    print(f'  {site:<12}: done\n')

## 4. Spectral Indices — Sangatta

In [ ]:
# ============================================================
# Load processed bands and compute 5 indices
# ============================================================
SITE     = 'sangatta'
SCENE_ID = SCENES[SITE]

data    = load_geotiff_bands(str(DATA_PROC), SCENE_ID, site=SITE)
indices = compute_all_indices(data)

print('\nIndex statistics:')
for name, arr in indices.items():
    print(f'  {name:<8}: min={np.nanmin(arr):.3f}  max={np.nanmax(arr):.3f}  mean={np.nanmean(arr):.3f}')

In [ ]:
# ============================================================
# Band statistics and index distribution check
# ============================================================
import rasterio

tif_path = DATA_PROC / f'{SITE}_{SCENE_ID}_bands.tif'
with rasterio.open(tif_path) as src:
    for i in range(1, src.count + 1):
        tag = src.tags(i)
        arr = src.read(i)
        valid = arr[np.isfinite(arr)]
        print(f'  Band {i} ({tag.get("name","?"):<8}): '
              f'min={valid.min():.4f}  max={valid.max():.4f}  mean={valid.mean():.4f}')

print()
savi = indices['SAVI']
print(f'  SAVI NaN count  : {np.isnan(savi).sum():,}')
print(f'  SAVI zero count : {(savi == 0).sum():,}')
print(f'  SAVI < 0 count  : {(savi < 0).sum():,}')


In [ ]:
# ============================================================
# Diagnostic plot — all 5 indices
# ============================================================
# MVI has wider range [-1, 20] — use per-index vmin/vmax
VRANGE = {
    'NDMI'  : (-1, 1),
    'MNDWI' : (-1, 1),
    'MVI'   : (-1, 20),
    'SAVI'  : (-1, 1),
    'EMI'   : (-1, 1),
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, arr) in zip(axes, indices.items()):
    vmin, vmax = VRANGE.get(name, (-1, 1))
    im = ax.imshow(arr, cmap='RdYlGn', vmin=vmin, vmax=vmax)
    ax.set_title(name)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle(f'Spectral Indices — Sangatta ({SCENE_ID})', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'figures' / 'indices_sangatta.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Zoom to coastal area (land-water boundary)
mvi = indices['MVI']
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(mvi[300:600, 400:700], cmap='RdYlGn', vmin=-1, vmax=20)
ax.set_title('MVI - coastal zoom')
plt.show()

## 5. Adaptive Threshold Calibration — Sangatta

In [ ]:
# ============================================================
# Coastal candidate mask -- adaptive per scene
# All thresholds derived per-scene via Otsu (no fixed parameters)
# ============================================================
candidate_mask, candidate_diag = compute_coastal_candidate_mask(
    data, indices,
    max_buffer_m=500.0,   # hard cap: mangrove constrained to tidal zone
    return_diagnostics=True
)

# Save candidate mask as GeoTIFF for use in 02_classification.ipynb
import rasterio
cand_path = DATA_PROC / f'candidate_{SITE}_{SCENE_ID}.tif'
with rasterio.open(
    cand_path, 'w',
    driver='GTiff',
    height=candidate_mask.shape[0],
    width=candidate_mask.shape[1],
    count=1, dtype='uint8',
    crs=data['crs'], transform=data['transform'],
    compress='lzw'
) as dst:
    dst.write(candidate_mask.astype(np.uint8), 1)
print(f'Candidate mask saved : {cand_path.name}')

In [ ]:
# ============================================================
# Diagnostic plot -- candidate mask components
# ============================================================
capped_label = '  [capped]' if candidate_diag['buffer_capped'] else ''

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(candidate_diag['water_mask'], cmap='Blues')
axes[0].set_title(f'Water mask (MNDWI > {candidate_diag["water_t"]:.3f})')
axes[0].axis('off')

axes[1].imshow(candidate_diag['veg_mask'], cmap='Greens')
axes[1].set_title(f'Vegetation mask (SAVI > {candidate_diag["veg_t"]:.3f})')
axes[1].axis('off')

axes[2].imshow(candidate_diag['dist_px'], cmap='magma', vmax=candidate_diag['buffer_t_px']*3)
axes[2].set_title(f'Distance from water (px)\n'
                  f'otsu={candidate_diag["buffer_otsu_px"]:.0f} px  '
                  f'final={candidate_diag["buffer_t_px"]:.0f} px (~{candidate_diag["buffer_t_m"]:.0f} m)'
                  f'{capped_label}')
axes[2].axis('off')

axes[3].imshow(candidate_mask, cmap='YlGn')
axes[3].set_title('Coastal candidate zone')
axes[3].axis('off')

plt.suptitle(f'Adaptive Coastal Candidate Mask -- {SITE}', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'figures' / f'candidate_mask_{SITE}.png',
            dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# ============================================================
# Per-scene adaptive threshold calibrated within candidate zone
# MVI uses Otsu directly (right-skewed distribution -- bimodal
# valley detection picks an overly strict threshold for MVI)
# ============================================================
import json

thresholds = apply_adaptive_threshold(
    indices, scene_id=SCENE_ID,
    candidate_mask=candidate_mask,
    force_otsu_indices=['MVI'],
)

thresh_path = ROOT / 'outputs' / 'results' / f'thresholds_{SITE}_{SCENE_ID}.json'
with open(thresh_path, 'w') as f:
    json.dump(
        {k: float(v) for k, v in thresholds.items()
         if v is not None and np.isfinite(v)}, f, indent=2
    )
print(f'Thresholds saved : {thresh_path.name}')


In [ ]:
# ============================================================
# Diagnostic plot -- histogram + threshold lines for MVI + NDMI
# Histogram restricted to candidate zone (consistent with how
# thresholds were calibrated)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, name in zip(axes, ['MVI', 'NDMI']):
    arr  = indices[name]
    flat = arr[candidate_mask & np.isfinite(arr)].ravel()
    ax.hist(flat, bins=256, color='steelblue', alpha=0.7)
    if thresholds.get(name) is not None:
        ax.axvline(thresholds[name], color='red', linewidth=2,
                   label=f'threshold = {thresholds[name]:.4f}')
    ax.set_title(f'{name} -- Bimodal Histogram (candidate zone)')
    ax.set_xlabel('Index value')
    ax.set_ylabel('Pixel count')
    ax.legend()

plt.suptitle(f'Adaptive Thresholds -- Sangatta (candidate zone, n={candidate_mask.sum():,} px)', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'figures' / 'thresholds_sangatta.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 6. Hyperspectral Diagnostic Feature

Tanager's 426-band spectrum enables a physically-grounded feature that broadband multispectral sensors (Sentinel-2, Landsat) cannot reproduce. It is used as an additional input to the RF/XGBoost classifier. The adaptive threshold and pseudo-labels remain based on the 5 spectral indices only.

### 6a. Red-edge Inflection Point (REIP)

Wavelength of maximum first derivative in the red-edge region (670-760 nm). Mangrove shows a sharp sigmoid rise (~660-760 nm) due to high chlorophyll content, while non-mangrove surfaces are flatter. REIP requires ~19 narrow contiguous bands and cannot be derived from broadband sensors.

In [ ]:
# ============================================================
# Load full 426-band HDF5 spectrum for the current scene
# REIP needs the full cube, so use load_hdf5() (not the 5-band GeoTIFF)
# ============================================================
hdf5_data = load_hdf5(str(h5_path(SITE)))          # full 426-band spectrum
print(f"HDF5 loaded : {hdf5_data['reflectance'].shape[0]} bands")

In [ ]:
# ============================================================
# Compute REIP from full HDF5 spectrum (red-edge 670-760nm)
# ============================================================
reip_map  = compute_reip(hdf5_data, left_nm=670, right_nm=760)

reip_path = DATA_PROC / f'reip_{SITE}_{SCENE_ID}.tif'
with rasterio.open(
    reip_path, 'w', driver='GTiff',
    height=reip_map.shape[0], width=reip_map.shape[1],
    count=1, dtype='float32',
    crs=hdf5_data['crs'], transform=hdf5_data['transform'],
    compress='lzw', nodata=np.nan,
) as dst:
    dst.write(reip_map, 1)
print(f'REIP saved : {reip_path.name}')

## 7. Full Pipeline Loop — All Sites

Runs the complete pipeline (HDF5 conversion, indices, candidate mask, adaptive threshold) for all 5 transfer sites. The hyperspectral feature (REIP) is computed per site from the full HDF5 spectrum.

In [ ]:
# ============================================================
# Full pipeline loop — all sites
# Convert HDF5 → GeoTIFF → indices → adaptive threshold → save
# Skip conversion if multiband GeoTIFF already exists
# ============================================================
import json

all_thresholds = {}

for site, scene_id in SCENES.items():
    print(f'\n{"="*50}')
    print(f'  Site: {site.upper()}  ({scene_id})')
    print(f'{"="*50}')

    hp       = h5_path(site)
    tif_path = DATA_PROC / f'{site}_{scene_id}_bands.tif'

    # Step 1: Convert HDF5 → multiband GeoTIFF (skip if exists)
    if tif_path.exists():
        print(f'  GeoTIFF exists — skipping conversion')
    else:
        if not hp.exists():
            print(f'  SKIP — HDF5 not found')
            continue
        hdf5_to_geotiff(str(hp), str(DATA_PROC), scene_id, site=site)

    # Step 2: Load bands
    data = load_geotiff_bands(str(DATA_PROC), scene_id, site=site)

    # Step 3: Compute indices
    indices = compute_all_indices(data)

    # Step 4: Adaptive threshold
    candidate_mask = compute_coastal_candidate_mask(
        data, indices,
        max_buffer_m=500.0,   # hard cap consistent across all sites
    )

    # Save candidate mask
    cand_path = DATA_PROC / f'candidate_{site}_{scene_id}.tif'
    with rasterio.open(
        cand_path, 'w', driver='GTiff',
        height=candidate_mask.shape[0], width=candidate_mask.shape[1],
        count=1, dtype='uint8',
        crs=data['crs'], transform=data['transform'], compress='lzw'
    ) as dst:
        dst.write(candidate_mask.astype(np.uint8), 1)

    # Step 4b: Adaptive threshold (constrained to candidate zone)
    thresholds = apply_adaptive_threshold(
        indices, scene_id=scene_id, candidate_mask=candidate_mask,
        force_otsu_indices=['MVI'],
    )
    all_thresholds[site] = thresholds

    # Step 5: Save threshold JSON per site
    thresh_path = ROOT / 'outputs' / 'results' / f'thresholds_{site}_{scene_id}.json'
    with open(thresh_path, 'w') as f:
        json.dump({k: float(v) for k, v in thresholds.items() if v is not None}, f, indent=2)
    print(f'  Thresholds saved : {thresh_path.name}')

    # Step 6: Hyperspectral diagnostic feature (full HDF5 spectrum)
    if hp.exists():
        hdf5_data_site = load_hdf5(str(hp))

        reip      = compute_reip(hdf5_data_site, left_nm=670, right_nm=760)
        reip_path = DATA_PROC / f'reip_{site}_{scene_id}.tif'
        with rasterio.open(
            reip_path, 'w', driver='GTiff',
            height=reip.shape[0], width=reip.shape[1],
            count=1, dtype='float32',
            crs=hdf5_data_site['crs'], transform=hdf5_data_site['transform'],
            compress='lzw', nodata=np.nan,
        ) as dst:
            dst.write(reip, 1)
        print(f'  REIP saved : {site}')
    else:
        print(f'  HDF5 not found -- skipping hyperspectral feature for {site}')

print(f'\nDone — {len(all_thresholds)} sites processed.')

In [ ]:
print("HDF5 shape    :", hdf5_data['reflectance'].shape[1:])   # (850, 810)
print("reip_map shape:", reip_map.shape)                       # follows HDF5
print("indices shape :", indices['NDMI'].shape)                # (667, 915)